In [1]:
from datetime import date
import polars as pl

# 動作設定
valuation_date = date(2025, 6, 1)
min_updown_rate = 10
display_record_num = 10

# 設定
pl.Config.set_tbl_rows(display_record_num)

polars.config.Config

In [2]:
# pathの設定
from pathlib import Path
import os

CURRENT_DIR = Path(os.getcwd())
PJ_DIR = CURRENT_DIR.parent.parent.parent
WS_DIR = PJ_DIR.parent



# import objects
from wequant.data_processing import read_data, KessanPl, CommonPl, FinancequotePl


# pl.Config
pl.Config.set_tbl_rows(display_record_num)

# read data
DATA_DIR = PJ_DIR / "data"
fp = DATA_DIR / "kessan.parquet"
kessan_df = read_data(fp)

# generate instance
KPL = KessanPl(kessan_df)
original_KPL = KPL

In [3]:
#####
##### 加工
#####

In [4]:
# データ加工1
# 評価日における最新四半期決算の取得
KPL = original_KPL
target_df = KPL.get_latest_quater_settlements(valuation_date)

# 騰落率列を追加する
target_KPL = KessanPl(target_df)
KPL = target_KPL
KPL.with_columns_updown_rate_on_announcement_date()

# 騰落率列("updown_rate_on_sett"列)がnull値のレコードを削除
KPL.filter_null("updown_rate_on_sett")
df = KPL.df
df = df.sort("updown_rate_on_sett", descending=True)
all_df = df

# updown_rate_on_settの騰落率がmin_updown_rate以上のレコードを抽出
df = all_df
df = df.filter(pl.col("updown_rate_on_sett")>=min_updown_rate)

target_meigara_df = df

In [5]:
###
### 解析
###

In [6]:
# 全体傾向
全銘柄数 = all_df.shape[0]

# 決算発表日における値上銘柄数
df = all_df
df = df.filter(pl.col("updown_rate_on_sett")>0)
値上銘柄数 = df.shape[0]
値上銘柄割合 = 100 * 値上銘柄数 / 全銘柄数
値上銘柄割合 = round(値上銘柄割合, 2)

# 決算発表日における騰落率平均(各銘柄騰落率の単純平均)
CPL = CommonPl(all_df)
columns=['updown_rate_on_sett']
df = CPL.get_aggregate_function_result(
    columns=columns,
    agg_func = "mean"
)
決算発表日騰落率平均 = round(df.row(0)[0], 2)

# 決算発表日騰落率閾値越
閾値越銘柄数 = target_meigara_df.shape[0]
閾値越割合 = 100 * (閾値越銘柄数 / 全銘柄数)

# 出力
print("***** 全体傾向 *****")
print(f'全銘柄数: {全銘柄数}')
print(f'決算発表日値上銘柄数: {値上銘柄数} ({round(値上銘柄割合, 2)}%)')
print(f'決算発表日騰落率平均: {決算発表日騰落率平均}%')
print()
print(f'*** 閾値{min_updown_rate}%超銘柄数')
print(f'決算発表日騰落率{min_updown_rate}%超銘柄数: {閾値越銘柄数} ({round(閾値越割合, 1)}%)')

***** 全体傾向 *****
全銘柄数: 3654
決算発表日値上銘柄数: 1711 (46.83%)
決算発表日騰落率平均: 0.21%

*** 閾値10%超銘柄数
決算発表日騰落率10%超銘柄数: 206 (5.6%)


In [7]:
# 発表決算の増収増益の比率を全体とターゲットで比較する
##### 全体
DATA_DIR = PJ_DIR / "data"
fp = DATA_DIR / "kessan.parquet"
df = read_data(fp)
df = df.filter(pl.col("settlement_type")=="四")
KPL = KessanPl(df)
KPL.with_columns_growth_rate()
df = KPL.get_latest_quater_settlements(valuation_date)
KPL.df = df
KPL.with_columns_updown_rate_on_announcement_date()
KPL.filter_null("updown_rate_on_sett")

original_df = KPL.df

df = original_df
全銘柄数 = original_df.shape[0]

# 増収増益(経常利益)
dff = df.filter(pl.col("gr_sales")>0).filter(pl.col("gr_ordinary_profit")>0)
全体増収増益数 = dff.shape[0]
全体増収増益割合 = 100 * 全体増収増益数/全銘柄数
全体増収増益割合 = round(全体増収増益割合, 2)

# 増収減益(経常利益)
dff = df.filter(pl.col("gr_sales")>0).filter(pl.col("gr_ordinary_profit")<0)
全体増収減益数 = dff.shape[0]
全体増収減益割合 = 100 * 全体増収減益数/全銘柄数
全体増収減益割合 = round(全体増収減益割合, 2)

# 減収増益(経常利益)
dff = df.filter(pl.col("gr_sales")<0).filter(pl.col("gr_ordinary_profit")>0)
全体減収増益数 = dff.shape[0]
全体減収増益割合 = 100 * 全体減収増益数/全銘柄数
全体減収増益割合 = round(全体減収増益割合, 2)

# 減収減益(経常利益)
dff = df.filter(pl.col("gr_sales")<0).filter(pl.col("gr_ordinary_profit")<0)
全体減収減益数 = dff.shape[0]
全体減収減益割合 = 100 * 全体減収減益数/全銘柄数
全体減収減益割合 = round(全体減収減益割合, 2)

##### ターゲット銘柄
df = original_df

# ターゲット銘柄に絞りこむ
target_codes = target_meigara_df["code"].to_list()
df = df.filter(pl.col("code").is_in(target_codes))
対象銘柄数 = df.shape[0]

# 増収増益(経常利益)
dff = df.filter(pl.col("gr_sales")>0).filter(pl.col("gr_ordinary_profit")>0)
対象銘柄増収増益数 = dff.shape[0]
対象銘柄増収増益割合 = 100 * 対象銘柄増収増益数/対象銘柄数
対象銘柄増収増益割合 = round(対象銘柄増収増益割合, 2)

# 増収減益(経常利益)
dff = df.filter(pl.col("gr_sales")>0).filter(pl.col("gr_ordinary_profit")<0)
対象銘柄増収減益数 = dff.shape[0]
対象銘柄増収減益割合 = 100 * 対象銘柄増収減益数/対象銘柄数
対象銘柄増収減益割合 = round(対象銘柄増収減益割合, 2)

# 減収増益(経常利益)
dff = df.filter(pl.col("gr_sales")<0).filter(pl.col("gr_ordinary_profit")>0)
対象銘柄減収増益数 = dff.shape[0]
対象銘柄減収増益割合 = 100 * 対象銘柄減収増益数/対象銘柄数
対象銘柄減収増益割合 = round(対象銘柄減収増益割合, 2)

# 減収減益(経常利益)
dff = df.filter(pl.col("gr_sales")<0).filter(pl.col("gr_ordinary_profit")<0)
対象銘柄減収減益数 = dff.shape[0]
対象銘柄減収減益割合 = 100 * 対象銘柄減収減益数/対象銘柄数
対象銘柄減収減益割合 = round(対象銘柄減収減益割合, 2)

### 出力
print(f'***** 発表決算の増収経常増益の比率を全体とターゲットで比較')
print(f'******* 全体({original_df.shape[0]}銘柄)')
print(f'増収経常増益: {全体増収増益数}銘柄({全体増収増益割合}%)')
print(f'増収経常減益: {全体増収減益数}銘柄({全体増収減益割合}%)')
print(f'減収経常増益: {全体減収増益数}銘柄({全体減収増益割合}%)')
print(f'減収経常減益: {全体減収減益数}銘柄({全体減収減益割合}%)')

### 出力
print()
print(f'******* 対象銘柄({df.shape[0]}銘柄)')
print(f'増収経常増益: {対象銘柄増収増益数}銘柄({対象銘柄増収増益割合}%)')
print(f'増収経常減益: {対象銘柄増収減益数}銘柄({対象銘柄増収減益割合}%)')
print(f'減収経常増益: {対象銘柄減収増益数}銘柄({対象銘柄減収増益割合}%)')
print(f'減収経常減益: {対象銘柄減収減益数}銘柄({対象銘柄減収減益割合}%)')

***** 発表決算の増収経常増益の比率を全体とターゲットで比較
******* 全体(3615銘柄)
増収経常増益: 1484銘柄(41.05%)
増収経常減益: 1123銘柄(31.07%)
減収経常増益: 253銘柄(7.0%)
減収経常減益: 741銘柄(20.5%)

******* 対象銘柄(203銘柄)
増収経常増益: 121銘柄(59.61%)
増収経常減益: 47銘柄(23.15%)
減収経常増益: 9銘柄(4.43%)
減収経常減益: 25銘柄(12.32%)


In [8]:
# ターゲット銘柄のうち、増収増益銘柄を深堀してみる

# ターゲット銘柄に絞りこむ
target_codes = target_meigara_df["code"].to_list()
df = original_df.filter(pl.col("code").is_in(target_codes))
決算発表日騰落率閾値越_増収増益_df = df.filter(pl.col("gr_sales")>0).filter(pl.col("gr_ordinary_profit")>0)

# 前年同期成長率の平均をとってみる
df = 決算発表日騰落率閾値越_増収増益_df
CPL = CommonPl(df)
増収増益_各指標の成長率の平均_result_df = CPL.get_aggregate_function_result(["gr_sales", "gr_operating_income", "gr_ordinary_profit", "gr_final_profit"], "mean")
増収増益_各指標の成長率の中央値_result_df = CPL.get_aggregate_function_result(["gr_sales", "gr_operating_income", "gr_ordinary_profit", "gr_final_profit"], "median")

# 出力
# 成長率の平均
print(f'各指標の成長率の平均')
print(増収増益_各指標の成長率の平均_result_df)

# 成長率の中央値
print(f'各指標の成長率の平均')
print(増収増益_各指標の成長率の中央値_result_df)

各指標の成長率の平均
shape: (1, 4)
┌──────────┬─────────────────────┬────────────────────┬─────────────────┐
│ gr_sales ┆ gr_operating_income ┆ gr_ordinary_profit ┆ gr_final_profit │
│ ---      ┆ ---                 ┆ ---                ┆ ---             │
│ f64      ┆ f64                 ┆ f64                ┆ f64             │
╞══════════╪═════════════════════╪════════════════════╪═════════════════╡
│ 21.64    ┆ 47.88               ┆ 163.28             ┆ 35.01           │
└──────────┴─────────────────────┴────────────────────┴─────────────────┘
各指標の成長率の平均
shape: (1, 4)
┌──────────┬─────────────────────┬────────────────────┬─────────────────┐
│ gr_sales ┆ gr_operating_income ┆ gr_ordinary_profit ┆ gr_final_profit │
│ ---      ┆ ---                 ┆ ---                ┆ ---             │
│ f64      ┆ f64                 ┆ f64                ┆ f64             │
╞══════════╪═════════════════════╪════════════════════╪═════════════════╡
│ 14.87    ┆ 51.73               ┆ 52.63              ┆ 50.0  

In [9]:
# 決算発表日前日のfinance_quoteを取得する
fp = DATA_DIR / "finance_quote.parquet"
df = read_data(fp)
FPL = FinancequotePl(df)

決算発表日騰落率閾値越_増収増益_codes = 決算発表日騰落率閾値越_増収増益_df["code"].to_list()
df = df.filter(pl.col("mcode").is_in(決算発表日騰落率閾値越_増収増益_codes))

# getするDataFrameの初期化
schema = {name: FPL.df.select(name).dtypes[0] for name in FPL.df.columns}
result_df = pl.DataFrame([], schema=schema)
xdf = 決算発表日騰落率閾値越_増収増益_df
for row in xdf.iter_rows(named=True):
    xcode = row["code"]
    xtarget_date = row["announcement_date"]
    row_df = FPL.get_meigara_lastdate_finance_quote(xcode, xtarget_date)
    result_df = result_df.vstack(row_df)

決算発表日騰落率閾値越_増収増益_決算発表前日ファンダメンタルズ_df = result_df

In [10]:
決算発表日騰落率閾値越_増収増益_df

code,announcement_date,settlement_date,settlement_type,sales,operating_income,ordinary_profit,final_profit,reviced_eps,dividend,quater,gr_sales,gr_operating_income,gr_ordinary_profit,gr_final_profit,updown_rate_on_sett
i64,date,date,str,i64,i64,i64,i64,f64,f64,i64,f64,f64,f64,f64,f64
1777,2025-04-28,2025-03-31,"""四""",10764,1580,1576,1214,101.5,14.7,4,56.75,159.87,156.68,237.22,20.95
1870,2025-05-07,2025-03-31,"""四""",43590,5336,5242,3325,77.3,12.2,4,50.29,1028.12,1110.62,1529.9,12.4
1884,2025-05-14,2025-03-31,"""四""",43451,2816,2798,1907,43.4,6.5,4,2.99,15.55,13.05,29.82,16.02
1946,2025-04-28,2025-03-31,"""四""",72653,5978,5646,4711,50.6,8.2,4,3.21,0.57,53.22,18.13,12.56
1951,2025-05-09,2025-03-31,"""四""",231552,22684,22593,15378,73.9,9.8,4,9.89,19.44,26.01,69.62,14.54
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
9441,2025-05-09,2025-03-31,"""四""",34250,2455,2523,1720,89.4,7.2,1,11.96,33.35,35.21,22.42,15.9
9627,2025-03-10,2025-01-31,"""四""",121974,6682,7281,4145,118.3,5.5,3,18.84,20.27,27.47,19.35,10.05
9628,2025-05-08,2025-03-31,"""四""",11215,2034,1871,3250,159.0,18.1,4,77.48,66.72,52.73,355.82,10.53


In [11]:
FPL.df

code,date,total_shares_num,expected_dividend_yield,expected_dividend_per_share,expected_PER,actual_PBR,expected_EPS,actual_BPS,actual_CAR,next_settlement_date,last_settlement_date
i64,date,f64,f64,f64,f64,f64,f64,f64,f64,date,date
1301,2024-07-05,1.20783e7,2.72,110.0,6.86,0.81,589.35,4965.39,36.7,2025-03-31,2025-03-31
1301,2024-07-19,1.20783e7,2.69,110.0,6.94,0.82,589.35,4965.39,36.7,2025-03-31,2025-03-31
1301,2024-07-22,1.20783e7,2.71,110.0,6.88,0.82,589.35,4965.39,36.7,2025-03-31,2025-03-31
1301,2024-07-23,1.20783e7,2.72,110.0,6.87,0.82,589.35,4965.39,36.7,2025-03-31,2025-03-31
1301,2024-07-24,1.20783e7,2.76,110.0,6.77,0.8,589.35,4965.39,36.7,2025-03-31,2025-03-31
…,…,…,…,…,…,…,…,…,…,…,…
9997,2025-08-04,9.72445e7,3.16,30.0,9.62,0.66,98.72,null,45.2,2026-03-31,null
9997,2025-08-05,9.72445e7,3.13,30.0,9.7,0.66,98.72,null,45.2,2026-03-31,null
9997,2025-08-06,9.72445e7,3.13,30.0,9.71,0.66,98.72,null,45.2,2026-03-31,null
